In [ ]:
"""
Лабораторная работа 4.2 — Вариант 28: Издательство
Наследование, Полиморфизм, Инкапсуляция
"""

# ======================================================
#  БАЗОВЫЙ КЛАСС 1 — Публикация (сущность)
# ======================================================

class Publication:
    """Любая публикация издательства: книга или журнал."""

    def __init__(self, title, circulation, price):
        self.title = title          # название
        self.circulation = circulation  # тираж (штук)
        self.price = price          # цена за экземпляр (руб.)

    def calculate_royalties(self):
        """
        Полиморфный метод — гонорар автора.
        Каждый дочерний класс считает его по-своему.
        """
        raise NotImplementedError("Метод не реализован в дочернем классе!")

    def __str__(self):
        return f"«{self.title}» | тираж: {self.circulation} шт. | цена: {self.price} руб."


# ======================================================
#  ДОЧЕРНИЙ КЛАСС 1А — Книга
#  Гонорар = 15% от (тираж × цена)
# ======================================================

class Book(Publication):
    """Книга с указанием жанра."""

    def __init__(self, title, circulation, price, genre):
        super().__init__(title, circulation, price)  # берём поля родителя
        self.genre = genre  # добавляем своё поле

    def calculate_royalties(self):
        return self.circulation * self.price * 0.15  # 15%

    def __str__(self):
        return f"Книга {super().__str__()} | жанр: {self.genre}"


# ======================================================
#  ДОЧЕРНИЙ КЛАСС 1Б — Журнал
#  Гонорар = 8% от (тираж × цена)
# ======================================================

class Magazine(Publication):
    """Журнал с указанием периодичности выхода."""

    def __init__(self, title, circulation, price, frequency):
        super().__init__(title, circulation, price)
        self.frequency = frequency  # например: "ежемесячный"

    def calculate_royalties(self):
        return self.circulation * self.price * 0.08  # 8%

    def __str__(self):
        return f"Журнал {super().__str__()} | выход: {self.frequency}"


# ======================================================
#  БАЗОВЫЙ КЛАСС 2 — Автор (субъект)
# ======================================================

class Author:
    """Автор, с которым заключён контракт."""

    def __init__(self, name, contract_amount):
        self.name = name                        # имя автора
        self.contract_amount = contract_amount  # сумма контракта (руб.)

    def __str__(self):
        return f"Автор: {self.name} | контракт: {self.contract_amount:,} руб."


# ======================================================
#  ДОЧЕРНИЙ КЛАСС 2 — Известный автор
#  Ему выплачивается аванс, который потом вычтется из гонорара
# ======================================================

class FamousAuthor(Author):
    """Известный автор, получивший аванс от издательства."""

    def __init__(self, name, contract_amount, advance):
        super().__init__(name, contract_amount)
        self.advance = advance  # аванс (руб.), уже выплачен заранее

    def __str__(self):
        return f"Известный автор: {self.name} | контракт: {self.contract_amount:,} руб. | аванс: {self.advance:,} руб."


# ======================================================
#  КЛАСС-КОНТЕЙНЕР — План издания
#  Связывает автора со списком его публикаций
# ======================================================

class PublishingPlan:
    """Издательский план: автор + список его публикаций."""

    def __init__(self, author):
        self.author = author   # объект Author или FamousAuthor
        self.items = []        # пустой список публикаций

    def add_item(self, publication):
        """Добавить публикацию в план."""
        self.items.append(publication)

    def calculate_total(self):
        """
        Считаем итоговый гонорар:
        — складываем гонорары всех публикаций (полиморфизм!)
        — если автор известный — вычитаем аванс, он уже был выплачен
        """
        total = sum(item.calculate_royalties() for item in self.items)

        if isinstance(self.author, FamousAuthor):
            total -= self.author.advance  # засчитываем аванс

        return total


# ======================================================
#  ГЛАВНЫЙ КОД — создание объектов и вывод результатов
# ======================================================

print("=" * 52)
print("   ЛР 4.2 — Издательство, Вариант 28")
print("=" * 52)


# --- Создаём авторов ---

print("\n--- Авторы ---\n")

author1 = Author("Андрей Белов", 200_000)
author2 = FamousAuthor("Мария Королёва", 500_000, advance=50_000)

print(author1)
print(author2)


# --- Создаём публикации ---

print("\n--- Публикации ---\n")

book1     = Book("Путь к вершинам Python", 10_000, 799, "Учебник")
book2     = Book("Тёмный лес",             30_000, 499, "Роман")
magazine1 = Magazine("Мир программирования",  5_000, 350, "ежемесячный")
magazine2 = Magazine("Литературный вестник",  15_000, 250, "еженедельный")

print(book1)
print(book2)
print(magazine1)
print(magazine2)


# --- Полиморфизм в действии ---
# Один и тот же метод calculate_royalties() —
# но Book считает 15%, а Magazine считает 8%

print("\n--- Полиморфизм: calculate_royalties() ---\n")

for pub in [book1, book2, magazine1, magazine2]:
    print(f"  {type(pub).__name__:10} «{pub.title}»  →  {pub.calculate_royalties():,.0f} руб.")


# --- Составляем планы издания ---

print("\n--- Планы издания ---\n")

plan1 = PublishingPlan(author1)
plan1.add_item(book1)
plan1.add_item(magazine1)

plan2 = PublishingPlan(author2)
plan2.add_item(book2)
plan2.add_item(magazine1)
plan2.add_item(magazine2)


# --- Итоговые отчёты ---

print("\n--- Отчёт: план обычного автора ---\n")
print(f"  Автор: {plan1.author.name}")
for item in plan1.items:
    print(f"  • {item.title:35} гонорар: {item.calculate_royalties():>12,.0f} руб.")
print(f"\n  ИТОГО К ВЫПЛАТЕ: {plan1.calculate_total():,.0f} руб.")

print("\n--- Отчёт: план известного автора ---\n")
print(f"  Автор: {plan2.author.name}  (аванс {plan2.author.advance:,} руб. будет засчитан)")
for item in plan2.items:
    print(f"  • {item.title:35} гонорар: {item.calculate_royalties():>12,.0f} руб.")
print(f"\n  Сумма без вычета аванса: {sum(i.calculate_royalties() for i in plan2.items):,.0f} руб.")
print(f"  Засчитывается аванс:    -{plan2.author.advance:,.0f} руб.")
print(f"\n  ИТОГО К ВЫПЛАТЕ: {plan2.calculate_total():,.0f} руб.")

   ЛР 4.2 — Издательство, Вариант 28

--- Авторы ---

Автор: Андрей Белов | контракт: 200,000 руб.
Известный автор: Мария Королёва | контракт: 500,000 руб. | аванс: 50,000 руб.

--- Публикации ---

Книга «Путь к вершинам Python» | тираж: 10000 шт. | цена: 799 руб. | жанр: Учебник
Книга «Тёмный лес» | тираж: 30000 шт. | цена: 499 руб. | жанр: Роман
Журнал «Мир программирования» | тираж: 5000 шт. | цена: 350 руб. | выход: ежемесячный
Журнал «Литературный вестник» | тираж: 15000 шт. | цена: 250 руб. | выход: еженедельный

--- Полиморфизм: calculate_royalties() ---

  Book       «Путь к вершинам Python»  →  1,198,500 руб.
  Book       «Тёмный лес»  →  2,245,500 руб.
  Magazine   «Мир программирования»  →  140,000 руб.
  Magazine   «Литературный вестник»  →  300,000 руб.

--- Планы издания ---


--- Отчёт: план обычного автора ---

  Автор: Андрей Белов
  • Путь к вершинам Python              гонорар:    1,198,500 руб.
  • Мир программирования                гонорар:      140,000 руб.

  ИТ